In [29]:
import pandas as pd
import plotly.graph_objects as go

lab_to_class = list(pd.read_csv("../../datasets/imagenet/classes.txt", header=None).values[0])

In [30]:
class_counts = pd.read_csv("../../experiment_data/feature_counts_imagenet/esug.csv").iloc[:, 1:]
class_counts = class_counts.rename(columns={"Count Majority": "maj", "Class": "cls", "Count": "val"})
class_counts["lab"] = class_counts["cls"].map(lambda x: lab_to_class.index(x))
class_counts.head()

,cls,val,maj,lab
0,tench,1,1,0
1,goldfish,3,2,1
2,great white shark,4,4,2
3,tiger shark,4,4,3
4,hammerhead,7,7,4


In [13]:
coverage = pd.read_csv("../../experiment_data/feature_counts_imagenet/esugCoverage.csv").iloc[:, 1:]
coverage = coverage.rename(columns={"Coverage": "cov", "True Class": "cls", "True Label": "lab", "Proportion": "prop", "Count": "size"})
coverage = coverage.sort_values("lab")
coverage.head()

,lab,cls,count,prop,cov
103,0,tench,658,0.002508,0.4874
164,1,goldfish,564,0.002150,0.4178
607,2,great white shark,72,0.000274,0.0533
779,3,tiger shark,31,0.000118,0.0230
876,4,hammerhead,23,0.000088,0.0170


In [14]:
valleys = pd.read_csv("../../experiment_data/feature_counts_imagenet/esugValleys.csv").iloc[:, 1:]
valleys = valleys.rename(columns={"major class size": "majsize", "majority class": "cls", "major class coverage": "majcov"}).drop(columns=["idx"])
valleys["lab"] = valleys["cls"].map(lambda x: lab_to_class.index(x))
valleys = valleys.sort_values("lab")
valleys.head()

,id,fstart,fend,ftype,pers,volume,logvol,cls,homogeneity,majsize,majcov,lab
5685,10974,2.384186e-07,0.009049,minima-saddle,0.002685,658,6.490724,tench,1.0,658,0.4874,0
873,1586,-0.000000e+00,0.009049,minima-saddle,0.000476,562,6.333280,goldfish,1.0,562,0.4163,1
2832,5396,1.566494e-03,0.010615,minima-saddle,0.001165,2,1.098612,goldfish,0.5,1,0.0007,1
376,647,4.339124e-05,0.009092,minima-saddle,0.000676,62,4.143135,great white shark,1.0,62,0.0459,2
4122,7933,9.989624e-03,0.019038,minima-saddle,0.004245,1,0.693147,great white shark,1.0,1,0.0007,2


How many classes have at least one corresponding valley where they are the majority? 

In [15]:
homo_val_owners = (class_counts["maj"] != 0).value_counts()[True]
homo_val_owners

np.int64(997)

In [18]:
valleys_homo_99 = valleys[valleys["homogeneity"] >= 0.99]
valleys_homo_95 = valleys[valleys["homogeneity"] >= 0.95]
valleys_homo_90 = valleys[valleys["homogeneity"] >= 0.90]

How many valleys are homogeneous?

In [19]:
valleys_homo_count_90 = len(valleys_homo_90)
valleys_homo_count_99 = len(valleys_homo_99)
valleys_homo_count_95 = len(valleys_homo_95)
print(f"{valleys_homo_count_90} / {len(valleys)}", f"{valleys_homo_count_95} / {len(valleys)}", f"{valleys_homo_count_99} / {len(valleys)}")

5810 / 5931 5809 / 5931 5800 / 5931


How many classes have at least 10% coverage in their homogeneous valleys?

In [24]:
valleys_homo_99.head()

,id,fstart,fend,ftype,pers,volume,logvol,cls,homogeneity,majsize,majcov,lab
5685,10974,2.384186e-07,0.009049,minima-saddle,0.002685,658,6.490724,tench,1.0,658,0.4874,0
873,1586,-0.000000e+00,0.009049,minima-saddle,0.000476,562,6.333280,goldfish,1.0,562,0.4163,1
376,647,4.339124e-05,0.009092,minima-saddle,0.000676,62,4.143135,great white shark,1.0,62,0.0459,2
4122,7933,9.989624e-03,0.019038,minima-saddle,0.004245,1,0.693147,great white shark,1.0,1,0.0007,2
4748,9123,3.060826e-04,0.009355,minima-saddle,0.000213,1,0.693147,great white shark,1.0,1,0.0007,2


In [25]:
pooled_cov = valleys_homo_99.groupby("lab").aggregate(majcov=('majcov', 'sum'), size=('majcov', 'size'))
maj_cov = (pooled_cov["majcov"] >= 0.1).value_counts()[True]

maj_cov, 1000 - maj_cov

(np.int64(521), np.int64(479))

5%? (65 / 1300 points)

In [26]:
pooled_cov = valleys_homo_99.groupby("lab").aggregate(majcov=('majcov', 'sum'), size=('majcov', 'size'))
maj_cov = (pooled_cov["majcov"] >= 0.05).value_counts()[True]

maj_cov, 1000 - maj_cov

(np.int64(596), np.int64(404))

2.5%? (33 / 1300 points)

In [28]:
pooled_cov = valleys_homo_99.groupby("lab").aggregate(majcov=('majcov', 'sum'), size=('majcov', 'size'))
maj_cov = (pooled_cov["majcov"] >= 0.025).value_counts()[True]

maj_cov, 1000 - maj_cov

(np.int64(734), np.int64(266))